# BirdCLEF 2026 — Submission Notebook
**Model:** EfficientNetB0 + transfer learning  
**Task:** For each 5-second audio window, predict the probability of presence for 234 species  
**Constraint:** CPU-only, max 90-minute runtime

## Cell 1 — Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')
print('TensorFlow version:', tf.__version__)

## Cell 2 — Paths and parameters configuration

In [ ]:
# ============================================================
# MODEL PATH
# REPLACE 'birdclef2026-model' with the exact name of your Kaggle dataset!
# ============================================================
MODEL_PATH = '/kaggle/input/birdclef2026-model/best_model.keras'

# PATH TO COMPETITION TEST DATA
TEST_AUDIO_DIR = '/kaggle/input/birdclef-2026/test_soundscapes'

# OUTPUT FILE PATH
SUBMISSION_PATH = '/kaggle/working/submission.csv'

# ============================================================
# AUDIO PARAMETERS — identical to those used during training!
# ============================================================
SAMPLE_RATE = 32000      # sampling frequency in Hz
DURATION = 5.0           # each audio window is 5 seconds long
N_MELS = 128             # number of mel bands (Y axis of spectrogram)
HOP_LENGTH = 512         # samples between successive spectrogram frames
N_FFT = 2048             # FFT window size
FMIN = 50                # minimum frequency (Hz)
FMAX = 14000             # maximum frequency (Hz)

# MODEL INPUT SIZE
# EfficientNetB0 expects (128, 313, 3) based on model.summary()
TARGET_HEIGHT = 128      # spectrogram height = N_MELS
TARGET_WIDTH = 313       # time frames: ceil(160000 / 512) + 1 = 313

print('Configuration loaded!')
print(f'  Model: {MODEL_PATH}')
print(f'  Test audio: {TEST_AUDIO_DIR}')
print(f'  Output: {SUBMISSION_PATH}')
print(f'  Spectrogram shape: ({TARGET_HEIGHT}, {TARGET_WIDTH}, 3)')

## Cell 3 — Load the list of 234 species (submission.csv columns)

In [ ]:
SAMPLE_SUB_PATH = '/kaggle/input/birdclef-2026/sample_submission.csv'

sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print('First 3 rows of sample_submission:')
print(sample_sub.head(3))
print(f'\nShape: {sample_sub.shape}')

# All columns except the first one ('row_id') are species names
SPECIES_LIST = list(sample_sub.columns[1:])

print(f'Number of species: {len(SPECIES_LIST)}')
print(f'First 5 species: {SPECIES_LIST[:5]}')

## Cell 4 — Load the model

In [ ]:
print('Loading model...')

model = tf.keras.models.load_model(MODEL_PATH, compile=False)

print('Model loaded successfully!')
print(f'  Input shape: {model.input_shape}')
print(f'  Output shape: {model.output_shape}')

## Cell 5 — Function: audio array → mel-spectrogram

In [ ]:
def audio_to_spectrogram(audio_array, sr=SAMPLE_RATE):
    """
    Converts a raw audio array into a mel-spectrogram ready for the model.

    Steps:
        1. Compute mel-spectrogram with librosa
        2. Convert to dB scale (log)
        3. Pad or truncate to TARGET_WIDTH frames
        4. Normalize to [0, 1]
        5. Stack into 3 channels for EfficientNet (RGB-like)

    Input:
        audio_array : 1D numpy array, shape (N,)
        sr          : sample rate (default 32000)

    Output:
        spectrogram : numpy array, shape (128, 313, 3), dtype float32
    """

    # Step 1: compute mel-spectrogram
    mel_spec = librosa.feature.melspectrogram(
        y=audio_array,
        sr=sr,
        n_mels=N_MELS,
        hop_length=HOP_LENGTH,
        n_fft=N_FFT,
        fmin=FMIN,
        fmax=FMAX
    )
    # mel_spec shape: (128, T)

    # Step 2: convert to dB scale
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

    # Step 3: pad or truncate to TARGET_WIDTH
    current_width = mel_spec_db.shape[1]
    if current_width < TARGET_WIDTH:
        mel_spec_db = np.pad(
            mel_spec_db,
            ((0, 0), (0, TARGET_WIDTH - current_width)),
            mode='constant'
        )
    elif current_width > TARGET_WIDTH:
        mel_spec_db = mel_spec_db[:, :TARGET_WIDTH]

    # Step 4: normalize to [0, 1]
    min_val = mel_spec_db.min()
    max_val = mel_spec_db.max()
    if max_val > min_val:
        mel_spec_norm = (mel_spec_db - min_val) / (max_val - min_val)
    else:
        mel_spec_norm = np.zeros_like(mel_spec_db)

    # Step 5: stack into 3 channels (EfficientNet expects RGB)
    mel_spec_rgb = np.stack([mel_spec_norm, mel_spec_norm, mel_spec_norm], axis=-1)
    # shape: (128, 313, 3)

    return mel_spec_rgb.astype(np.float32)


print('Function audio_to_spectrogram defined!')

## Cell 6 — Function: process a full audio file (split into 5s windows)

In [ ]:
def process_audio_file(audio_path):
    """
    Reads an audio file, splits it into 5-second windows,
    and returns a list of (row_id, spectrogram) tuples.

    BirdCLEF 2026 row_id format: 'FILENAME_ENDSECOND'
    Example: 'soundscape_001_5'  -> file soundscape_001, window ending at second 5
             'soundscape_001_10' -> window ending at second 10

    Input:
        audio_path : full path to the .ogg file

    Output:
        list of (row_id, spectrogram) tuples
    """

    filename = os.path.splitext(os.path.basename(audio_path))[0]
    results = []

    try:
        # Load the full audio file, resampled to SAMPLE_RATE, mono
        y, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)

        # Number of samples in 5 seconds: 5 * 32000 = 160000
        samples_per_window = int(DURATION * SAMPLE_RATE)

        # Number of complete 5-second windows
        n_windows = len(y) // samples_per_window

        for i in range(n_windows):
            start = i * samples_per_window
            end = start + samples_per_window
            window = y[start:end]

            spec = audio_to_spectrogram(window, sr=sr)

            # row_id: window ending second (5, 10, 15, ...)
            end_second = (i + 1) * int(DURATION)
            row_id = f'{filename}_{end_second}'

            results.append((row_id, spec))

    except Exception as e:
        print(f'  ERROR processing {filename}: {e}')

    return results


print('Function process_audio_file defined!')

## Cell 7 — Main inference loop

In [ ]:
# Find all .ogg test files
all_files = os.listdir(TEST_AUDIO_DIR)
audio_files = [f for f in all_files if f.endswith('.ogg')]
audio_files.sort()

print(f'Found {len(audio_files)} test audio files')

all_row_ids = []
all_predictions = []

BATCH_SIZE = 16
batch_specs = []
batch_ids = []

def flush_batch():
    """Run inference on the current batch and store results."""
    if not batch_specs:
        return
    X = np.array(batch_specs, dtype=np.float32)
    preds = model.predict(X, verbose=0)
    all_row_ids.extend(batch_ids)
    all_predictions.extend(preds)
    batch_specs.clear()
    batch_ids.clear()


# Main loop
for file_idx, filename in enumerate(audio_files):
    audio_path = os.path.join(TEST_AUDIO_DIR, filename)

    if file_idx % 10 == 0:
        print(f'Processing file {file_idx + 1}/{len(audio_files)}: {filename}')

    windows = process_audio_file(audio_path)

    for row_id, spec in windows:
        batch_ids.append(row_id)
        batch_specs.append(spec)
        if len(batch_specs) >= BATCH_SIZE:
            flush_batch()

# Process remaining windows
flush_batch()

print(f'\nInference complete!')
print(f'Total windows processed: {len(all_row_ids)}')

## Cell 8 — Build and save submission.csv

In [ ]:
predictions_array = np.array(all_predictions)
print(f'Predictions shape: {predictions_array.shape}')

submission_df = pd.DataFrame(
    predictions_array,
    columns=SPECIES_LIST
)
submission_df.insert(0, 'row_id', all_row_ids)

print('\nFirst 3 rows of submission:')
print(submission_df.head(3))
print(f'\nSubmission shape: {submission_df.shape}')

submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f'\nSubmission saved to: {SUBMISSION_PATH}')

## Cell 9 — Final verification

In [ ]:
check_df = pd.read_csv(SUBMISSION_PATH)
sample_sub_check = pd.read_csv(SAMPLE_SUB_PATH)

pred_values = check_df.iloc[:, 1:].values

print('=== SUBMISSION VERIFICATION ===')
print(f'Rows: {len(check_df)}')
print(f'Columns: {len(check_df.columns)} (expected 235 = 1 row_id + 234 species)')
print(f'First column: {check_df.columns[0]} (must be "row_id")')
print(f'Min value: {pred_values.min():.4f} (must be >= 0)')
print(f'Max value: {pred_values.max():.4f} (must be <= 1)')
print(f'Contains NaN: {np.isnan(pred_values).any()} (must be False)')
print(f'Columns match sample_submission: {list(check_df.columns) == list(sample_sub_check.columns)}')
print('\n Submission ready!')